# Calibrated Uncertainty under Image Perturbations for Galaxy Morphology Classification

**A walkthrough of the `galaxy-uq` project** &nbsp;—&nbsp; COMP90051 Statistical Machine Learning, 2026 S1.

---

### What this notebook is

A guided tour of the project end-to-end: dataset, the three classifiers we trained, the calibration metrics we measured, and the perturbation experiment that is the headline result. It is written to be readable by someone who knows the basics of statistical ML and a little linear algebra, but who has not seen *deep ensembles* or *expected calibration error* before. The accompanying report (`report/report.pdf`) is the four-page assignment artefact; this notebook is the long-form companion.

All figures shown here are produced by the scripts in `scripts/` and live in `results/figures/`. The notebook itself does not retrain the models — the full pipeline takes ~10 hours on an M-series laptop — it loads the saved JSON metrics and renders the saved PNGs. Every cell is reproducible from a clean checkout via `uv run python scripts/<NN>_*.py`.

### The research question

> *How does the predictive calibration of three classifiers of increasing complexity degrade as test images are progressively corrupted, and is a model's clean-data calibration ranking preserved under shift?*

Going beyond accuracy matters because downstream science pipelines consume class **probabilities**, not just top-1 labels. If a population study weights its likelihoods by classifier confidence, miscalibrated confidences corrupt the science.

### Table of contents

1. [Physics background — what is a galaxy morphology and why is it hard?](#1.-Physics-background)
2. [Statistical background — calibration, ECE, deep ensembles](#2.-Statistical-background)
3. [Dataset and exploratory analysis](#3.-Dataset-and-EDA)
4. [Feature engineering for the classical baselines](#4.-Features)
5. [Cross-validation protocol](#5.-Cross-validation)
6. [Model 1 — Logistic regression (from scratch)](#6.-Logistic-regression)
7. [Model 2 — Random forest](#7.-Random-forest)
8. [Model 3 — Deep ensemble CNN](#8.-Deep-ensemble-CNN)
9. [Calibration analysis](#9.-Calibration-analysis)
10. [Perturbation sweep — the headline result](#10.-Perturbation-sweep)
11. [Uncertainty decomposition under shift](#11.-Uncertainty-under-shift)
12. [Limitations](#12.-Limitations)
13. [Conclusion](#13.-Conclusion)

In [1]:
# Standard setup. All heavy compute lives in scripts/; this notebook only
# loads saved JSON metrics and embeds PNGs. Keeps the walkthrough cheap to
# re-render and the narrative reproducible from any clean checkout.
from pathlib import Path
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIG = ROOT / 'results' / 'figures'
MET = ROOT / 'results' / 'metrics'

def load(name):
    with open(MET / name) as f:
        return json.load(f)

## 1. Physics background

Galaxies, viewed from Earth, fall into a small number of visually distinct **morphological types**. The classical Hubble sequence splits them into ellipticals (smooth, featureless blobs of older stars), spirals (a central bulge with rotating arms of younger, bluer stars), and irregulars / mergers (disturbed shapes from recent gravitational interaction). The shape of a galaxy is a strong proxy for its star-formation history and dynamical state, which is why morphology is a useful axis along which to bin galaxies for population studies.

Classifying morphology is hard for three reasons that matter for ML:

1. **No canonical orientation.** A galaxy is just as much a galaxy when rotated by an arbitrary angle in the sky plane. Any model that is not built or trained to be rotation-invariant has to learn this from data.
2. **The classes are not crisp.** "Barred spiral" vs "unbarred tight spiral" is a continuum, and the [Galaxy Zoo](https://www.zooniverse.org/projects/zookeeper/galaxy-zoo/) labels we use are derived from majority votes of volunteer humans — the label itself is noisy.
3. **Observation conditions vary.** Atmospheric seeing, telescope optics, cosmic-ray hits, and the relative brightness of the galaxy against the sky all degrade the image. A model trained on clean cutouts may be used on noisier ones. *This is the regime our perturbation sweep simulates.*

The dataset is Galaxy10 DECaLS: 17,736 RGB images at 256×256 pixels with 10 morphology classes, curated from the DECaLS survey using Galaxy Zoo votes (Leung & Bovy, 2019).

## 2. Statistical background

This project is about **calibration**, not just accuracy, so the relevant statistical machinery is worth a careful read.

### 2.1 What does "calibration" mean?

A classifier outputs a probability vector $p \in \Delta^{K-1}$ for each input. We say it is **calibrated** if, among all inputs for which the model predicts the top class with confidence $c$, it is actually correct a fraction $c$ of the time:

$$\Pr\big(\hat{y}=y \;\big|\; \hat{p}=c\big) = c \quad \text{for all } c \in [0,1].$$

A model that always outputs `0.9` on inputs it gets right 90% of the time is well-calibrated. A model that outputs `0.99` on inputs it gets right only 70% of the time is *over-confident*; the converse is *under-confident*. Calibration is orthogonal to accuracy: a uniformly random predictor is perfectly calibrated (and useless), while a 99%-accurate model that always outputs `1.0` is badly miscalibrated.

### 2.2 Expected Calibration Error (ECE)

The quantity above is a conditional expectation, which we estimate by binning. Partition the unit interval into $B$ equal-width bins, group predictions by their max-softmax confidence into bin $b$, then

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|S_b|}{N} \,\big|\, \mathrm{acc}(S_b) - \mathrm{conf}(S_b)\,\big|,$$

where $\mathrm{acc}(S_b)$ is the empirical accuracy of bin $b$ and $\mathrm{conf}(S_b)$ is the mean predicted confidence in that bin (Guo et al., 2017). The **reliability diagram** plots $\mathrm{acc}(S_b)$ vs $\mathrm{conf}(S_b)$; perfect calibration is the diagonal. ECE is the sample-weighted vertical distance from that diagonal. We use $B=15$ throughout.

### 2.3 Why deep networks are usually miscalibrated

Modern deep networks trained to convergence on cross-entropy tend to be **over-confident** — they push the max-softmax close to 1 even when wrong (Guo et al., 2017). Intuitively: cross-entropy keeps decreasing as long as the model can sharpen its softmax, even after the argmax stops changing. Logistic regression with L2 penalty cannot do this: the convex objective converges to the proper-scoring-rule optimum and the linear head cannot inflate confidence arbitrarily. We will see this play out in the results.

### 2.4 Deep ensembles (Lakshminarayanan, Pritzel & Blundell, 2017)

A **deep ensemble** is, at its simplest, $M$ neural networks of the same architecture trained from $M$ different random initialisations on the same data, with prediction

$$\bar{p}(y \mid x) = \frac{1}{M}\sum_{m=1}^{M} p_m(y \mid x).$$

The members find different local optima of the (highly non-convex) loss landscape, so they make different mistakes; averaging their softmaxes both **improves accuracy** (variance reduction over correlated-but-imperfect predictors, the same intuition as bagging) and **dampens over-confidence** (the average of confident-and-wrong with confident-and-right is less confident than either). Lakshminarayanan et al. (2017) show on a wide range of tasks that this simple recipe is, in practice, the strongest baseline for predictive uncertainty in deep learning.

### 2.5 Decomposing predictive uncertainty

The total entropy of the ensemble prediction admits an exact decomposition:

$$\underbrace{H[\bar p]}_{\text{total}} \;=\; \underbrace{\tfrac{1}{M}\sum_m H[p_m]}_{\text{aleatoric (data noise)}} \;+\; \underbrace{H[\bar p] - \tfrac{1}{M}\sum_m H[p_m]}_{\text{epistemic (model disagreement)}}.$$

The first term is what every member thinks the irreducible noise is; the second is the gap between the average member's entropy and the entropy of the average, which by Jensen's inequality is non-negative and exactly zero iff all members agree. **Epistemic entropy is the signal that the model has not seen data like this before** — if every member confidently guesses a *different* class, $H[\bar p]$ is high but each $H[p_m]$ is low. This is the quantity we expect to spike under distribution shift, and it is the property we test in Section 11.

### 2.6 Temperature scaling

A cheap post-hoc fix for over-confidence: divide the logits by a scalar $T$ before softmax, and pick $T^\star$ that minimises NLL on a held-out calibration set (Guo et al., 2017). $T>1$ flattens, $T<1$ sharpens, $T=1$ is a no-op. Temperature scaling preserves the argmax (so accuracy is unchanged) and changes only the confidence. Its weakness, which we will demonstrate, is that $T^\star$ fit on clean data is not the right $T$ under distribution shift.

## 3. Dataset and EDA

Galaxy10 DECaLS: 17,736 images, 10 classes, moderate class imbalance (the rare *Cigar* class is <2% of the data; *Round Smooth* is ~15%). Here is a stratified random sample, one column per class:

![Sample grid](../results/figures/01_sample_grid.png)

*Two galaxies sampled from each class. The visual similarity across some pairs of classes (e.g. tight vs loose spiral, edge-on with/without bulge) is what makes the ten-way classification non-trivial even for human labellers.*

In [2]:
# Class counts (loaded directly from the EDA summary script's JSON output).
eda = load('01_eda_summary.json')
for k, v in eda['per_class_counts'].items():
    print(f'  {k:30s} {v:>5d}')
print(f'  {"TOTAL":30s} {eda["total_count"]:>5d}')

  Disturbed                       1081
  Merging                         1853
  Round Smooth                    2645
  In-between Round Smooth         2027
  Cigar Shaped Smooth              334
  Barred Spiral                   2043
  Unbarred Tight Spiral           1829
  Unbarred Loose Spiral           2628
  Edge-on without Bulge           1423
  Edge-on with Bulge              1873
  TOTAL                          17736


![Class distribution](../results/figures/01_class_distribution.png)

The 8× imbalance between Cigar (n=334) and Round Smooth (n=2,645) is the reason we (a) select hyperparameters on **macro-F1** rather than accuracy, and (b) use stratified folds throughout the cross-validation, so every fold contains a representative number of minority-class samples.

### Class-conditional pixel means

Averaging all training images within each class collapses to a remarkably interpretable picture of what each label *means* on average:

![Mean galaxy per class](../results/figures/01_mean_per_class.png)

The round-smooth and in-between classes average to a symmetric Gaussian-like blob; the edge-on classes show a clear horizontal streak; the spirals are nearly circularly symmetric on average because the arm positions cancel out across the sample. This is also a sanity check that the dataset is *centred* — the brightness peak of every class is at the image centre, confirming the cutouts were extracted around source coordinates.

## 4. Features

The two classical models (LR, RF) operate on a 113-dimensional handcrafted feature vector per image, split into six interpretable groups:

| Group | Dim | What it captures |
|---|---:|---|
| Photometric statistics | 16 | mean/std/skew/kurtosis of R, G, B, gray |
| CAS + Gini–M₂₀ | 10 | concentration, asymmetry, smoothness; Lotz et al. shape statistics |
| Radial profiles | 20 | 10-bin gray + 10-bin (G−R) colour-ratio profiles |
| Hu moments | 7 | log-transformed rotation/scale-invariant moments |
| HOG | 36 | 2×2 cell, 9-orientation histogram of oriented gradients |
| Gabor magnitude | 24 | mean+std response of a 3-freq × 4-orient filter bank |

The CAS+Gini–M₂₀ group is the astrophysics literature's preferred non-parametric morphology summary (Conselice 2003; Lotz et al. 2004) — these features were *designed* to separate galaxy morphologies, and we keep them so that the classical baselines have a fair shot.

![Key features by class](../results/figures/02_key_features_by_class.png)

Boxplots of the most discriminative handcrafted features. Asymmetry separates mergers and disturbed galaxies from smooth ellipticals; the Gini coefficient and concentration separate compact bulges from diffuse profiles.

![PCA of feature space](../results/figures/02_pca_2d.png)

Projecting the 113-dim feature space onto its first two principal components shows what a linear classifier has to work with: the smooth/round classes (yellows/oranges) and the edge-on classes (blues) form roughly separable lobes, but the spiral subclasses overlap heavily. This is foreshadowing for the per-class F1 results.

### Augmentation

For the CNN we apply per-sample horizontal flips and uniform 90° rotations during training. Galaxies have no canonical orientation, so multiples of 90° are **exact** label-preserving symmetries (no interpolation artefacts). This baked-in equivariance is the reason rotation by 90/180° will leave the CNN's accuracy basically unchanged in Section 10.

![Augmentation grid](../results/figures/02_augmentation_grid.png)

## 5. Cross-validation

All three models are evaluated under the same **nested 10×3 stratified cross-validation**, implemented from scratch in `src/galaxy_uq/cv.py`:

- **Outer loop:** 10-fold stratified split. Each test fold has ~1,774 samples; mean & std across the 10 folds are the headline numbers.
- **Inner loop:** 3-fold stratified split *inside each outer training partition*, used purely for hyperparameter selection.
- **Standardisation:** mean/std refit on the inner-train partition only — critical to avoid information leakage from test into HP selection.
- **Selection criterion:** macro-F1 (not accuracy), to keep the rare Cigar class influential in HP picks.

Why nested? A single train/val/test split conflates two questions ("which HPs are best?" and "how well does the model generalise?"). Nested CV answers them separately and gives proper uncertainty bars on the latter, at the cost of $10\times 3 = 30$ model fits per HP candidate. The compute is the price of an honest estimate.

## 6. Logistic regression

**The simplest baseline.** Multinomial softmax regression with L2 regularisation, implemented in pure NumPy and fit by full-batch gradient descent. The model is

$$p(y=k \mid x) = \frac{\exp(w_k^\top x + b_k)}{\sum_{j=1}^{K}\exp(w_j^\top x + b_j)},$$

trained by minimising the L2-penalised negative log-likelihood. We mirror scikit-learn's `C` parameterisation (where `C` is the inverse regularisation strength) and use a $1/N$-normalised gradient so the chosen `C` is directly comparable to sklearn references.

In [3]:
lr_res = load('03_logreg_results.json')
agg = lr_res['aggregate_metrics']
print('Logistic regression — nested 10x3 CV (mean ± std across 10 outer folds):')
for k, v in agg.items():
    print(f'  {k:10s} {v["mean"]:.4f} ± {v["std"]:.4f}')

Logistic regression — nested 10x3 CV (mean ± std across 10 outer folds):
  accuracy   0.4939 ± 0.0134
  macro_f1   0.4279 ± 0.0113
  ece        0.0660 ± 0.0114
  nll        1.4677 ± 0.0167


<table>
<tr><td><img src='../results/figures/03_logreg_hp_selection.png' width='100%'/></td>
<td><img src='../results/figures/03_logreg_reliability.png' width='100%'/></td></tr>
<tr><td><b>HP surface:</b> inner-CV macro-F1 vs C. Bowl-shaped → the modal pick C=10 is genuinely in the interior of the grid, not at a boundary.</td>
<td><b>Reliability:</b> LR's bins hug the diagonal closely. ECE = 0.066, the best of the three models.</td></tr>
</table>

![LR confusion matrix](../results/figures/03_logreg_confusion.png)

LR predicts the *common* classes acceptably and the spirals badly — unsurprising given the PCA view, where the spiral subclasses are not linearly separable in feature space.

## 7. Random forest

**The medium-complexity baseline.** Sklearn's `RandomForestClassifier` on the same 113 features, with `class_weight="balanced"` to compensate for the imbalance. The forest grows decorrelated trees on bootstrap samples; each split considers a random subset of features, which both reduces correlation between trees (so averaging reduces variance more) and provides an *implicit* feature-importance signal.

In [4]:
rf_res = load('04_rf_results.json')
agg = rf_res['aggregate_metrics']
print('Random forest — nested 10x3 CV:')
for k, v in agg.items():
    print(f'  {k:10s} {v["mean"]:.4f} ± {v["std"]:.4f}')

Random forest — nested 10x3 CV:
  accuracy   0.5114 ± 0.0167
  macro_f1   0.4603 ± 0.0077
  ece        0.1388 ± 0.0110
  nll        1.4569 ± 0.0342


![RF feature importance](../results/figures/04_rf_feature_importance.png)

The RF's permutation importances rank radial-profile bins, Gini, asymmetry and a handful of HOG cells as the most informative features — in line with the astrophysical intuition that shape statistics matter more than raw pixel statistics for morphology.

<table>
<tr><td><img src='../results/figures/04_rf_hp_selection.png' width='100%'/></td>
<td><img src='../results/figures/04_rf_reliability.png' width='100%'/></td></tr>
<tr><td><b>HP surface:</b> bowl-shaped in max_depth (modal pick 20 / 10–50); n_estimators saturates at the top of the grid — a structural property of bagging (variance reduction is monotone in tree count) rather than under-exploration.</td>
<td><b>Reliability:</b> RF is the most miscalibrated of the three, with characteristic <i>over-confidence</i> in the top bin — a known property of vote-based ensembles whose probability estimates are sharpened by the majority rule.</td></tr>
</table>

## 8. Deep ensemble CNN

**The headline model.** Five independent four-block VGG-style CNNs, each with the architecture

```
Conv–BN–ReLU → Pool   (× 3 blocks, channels 32 → 64 → 128)
 → AdaptiveAvgPool 2×2
 → Flatten → Dropout → Linear → ReLU → Dropout → Linear (10-way logits)
```

trained with AdamW, class-weighted cross-entropy, and a 10% stratified hold-out for early stopping (patience 5, max 20 epochs). The five members differ only in their random initialisation. At test time we average their softmaxes; that average **is** the deep ensemble's prediction (see §2.4 for the theory).

In [5]:
cnn_res = load('05_cnn_results.json')
agg = cnn_res['aggregate_metrics']
print('Deep ensemble CNN (M=5, 64x64) — nested 10x3 CV:')
for k, v in agg.items():
    print(f'  {k:10s} {v["mean"]:.4f} ± {v["std"]:.4f}')

Deep ensemble CNN (M=5, 64x64) — nested 10x3 CV:
  accuracy   0.6036 ± 0.0188
  macro_f1   0.5795 ± 0.0187
  ece        0.1365 ± 0.0159
  nll        1.1747 ± 0.0549


<table>
<tr><td><img src='../results/figures/05_cnn_training_curves.png' width='100%'/></td>
<td><img src='../results/figures/05_cnn_hp_selection.png' width='100%'/></td></tr>
<tr><td><b>Training curves</b> for one outer fold. Early stopping fires before any member overfits hard.</td>
<td><b>HP surface</b> over learning rate. Modal pick lr=1e-3 in 4/10 folds, with the neighbouring 3e-4 in another 4/10 — 80% of selections at or one step from the centre of the grid.</td></tr>
</table>

![Ensemble member disagreement](../results/figures/05_cnn_ensemble_disagreement.png)

*Disagreement between ensemble members on a held-out batch. When all five members vote the same class the bar is tall on one entry; when they split, the entropy of the average rises. This is the per-input epistemic signal at the heart of the deep-ensemble idea.*

![CNN confusion matrix](../results/figures/05_cnn_confusion.png)

The CNN substantially outperforms the classical baselines on every off-diagonal that matters: the spirals are no longer collapsed into one another, and the edge-on classes are largely disentangled.

## 9. Calibration analysis

### 9.1 Headline numbers

| Model | Accuracy | Macro-F1 | ECE | NLL |
|---|---|---|---|---|
| LR (113 handcrafted features) | 0.494 ± 0.013 | 0.428 ± 0.011 | **0.066 ± 0.011** | 1.468 ± 0.017 |
| RF (113 handcrafted features) | 0.511 ± 0.017 | 0.460 ± 0.008 | 0.139 ± 0.011 | 1.457 ± 0.034 |
| Deep Ensemble CNN (M=5, 64×64) | **0.604 ± 0.019** | **0.580 ± 0.019** | 0.137 ± 0.016 | **1.175 ± 0.055** |

**The CNN dominates on accuracy, macro-F1 and NLL — but the lowly logistic regression has the best ECE.** This is the first half of the story: clean-data calibration ranking and clean-data accuracy ranking are *not* the same.

### 9.2 Reliability diagrams side-by-side

![Reliability comparison](../results/figures/05b_reliability_comparison.png)

LR is closest to the diagonal across most bins. The CNN's top bin sits noticeably *above* the diagonal: it is over-confident when it is most confident. RF is structurally over-confident across the upper half of the confidence range — the worst of the three despite being more accurate than LR.

### 9.3 Per-class ECE — the aggregate hides two opposite failure modes

The CNN's aggregate ECE of 0.137 is a visual average of two opposite biases:

In [6]:
per_class = load('05g_per_class_ece.json')
print(f'{"Class":<22}{"ECE":>8}{"acc":>8}{"conf":>8}')
for row in per_class:
    print(f'{row["name"]:<22}{row["ece"]:>8.3f}{row["accuracy"]:>8.3f}{row["mean_confidence"]:>8.3f}')

Class                      ECE     acc    conf
Disturbed                0.073   0.334   0.399
Merging                  0.137   0.543   0.416
Round Smooth             0.299   0.855   0.556
In-between Round         0.235   0.672   0.439
Cigar                    0.223   0.716   0.493
Barred Spiral            0.122   0.496   0.381
Tight Spiral             0.255   0.663   0.425
Loose Spiral             0.129   0.282   0.355
Edge-on no Bulge         0.118   0.825   0.708
Edge-on w/ Bulge         0.160   0.713   0.580


![Per-class ECE](../results/figures/05g_per_class_ece.png)

Two things to notice:

1. **Round Smooth is *under-confident*** — mean confidence 0.56 but accuracy 0.86. Class-weighted training pushes the model away from confidently predicting majority classes, and on the largest class this swings into systematic under-confidence.
2. **Loose Spiral and Edge-on-with-Bulge have small over-confident top bins** (gaps of +0.94 on 1 sample and +0.78 on 11 samples respectively). When the CNN does commit to high confidence on a hard spiral subclass, it is often wrong.

The aggregate ECE plus an over/under-confidence split is much more honest than the single number alone. **For a downstream science pipeline, knowing *where* the model is miscalibrated matters more than knowing the global ECE.**

### 9.4 Risk–coverage and selective prediction

If we are willing to *abstain* on the least-confident inputs and forward only the top-confidence subset to a downstream pipeline, how does error rate trade off against coverage?

![Risk–coverage](../results/figures/05b_risk_coverage.png)

At ~50% coverage on the CNN's max-softmax, the error rate is roughly halved. Even with miscalibrated *probabilities*, the **ordering** of confidences is informative enough to support useful selective prediction.

![Abstention gain](../results/figures/05b_abstention_gain.png)

### 9.5 Failure gallery

Looking at the confidently-wrong and high-entropy failure cases is the qualitative complement to the aggregate calibration numbers:

![Failure gallery](../results/figures/05e_failure_gallery.png)

Many of the confidently-wrong cases are genuinely ambiguous human-call boundary cases (Loose Spiral vs Disturbed vs Merging) — not pure model failure but label noise leaking into the calibration signal.

![Boundary cases](../results/figures/05f_boundary_cases.png)

### 9.6 Temperature scaling on clean data

![Temperature-scaled calibration](../results/figures/05c_calibration_comparison.png)

Fitting $T^\star$ by minimising NLL on a clean held-out set gives $T^\star \approx 0.73 < 1$ — i.e. the ensemble is *under-confident* on aggregate (the class-weighted-loss explanation again), so temperature scaling *sharpens* it. ECE drops from 0.11 to 0.04 on clean data. **But we will see in Section 11 that this same $T^\star$ *hurts* calibration under distribution shift.**

## 10. Perturbation sweep

**The research question.** We retrain a smaller CNN ($32\times32$, $M=2$, 5 epochs) on outer-fold-0 train data and evaluate it on the same held-out test set under four corruption families at five severities each (a $4\times5$ grid of 20 conditions):

| Family | Severities | What it simulates |
|---|---|---|
| Gaussian blur | $\sigma \in \{0, 1, 2, 4, 8\}$ px | atmospheric seeing, telescope defocus |
| Additive Gaussian pixel noise | $\sigma \in \{0, 10, 25, 50, 100\}$ (0–255) | shot noise, read noise on faint sources |
| Centred square occlusion | $s \in \{0, 32, 64, 96, 128\}$ px | cosmic-ray hits, bright foreground stars |
| Rotation | $\theta \in \{0°, 15°, 45°, 90°, 180°\}$ | no canonical sky orientation |

The reduced model lets us run all 20 conditions on a laptop; the absolute accuracies are lower than the full ensemble in Table 9.1, but the trend across corruption strength is the quantity of interest.

![Perturbation summary](../results/figures/06_summary.png)

*The headline plot.* Accuracy and ECE for all three models across all four perturbation families. Read each panel as: "as I corrupt the input more, what happens to the model's accuracy and to its calibration?"

### 10.1 Rotation: the augmentation worked

![Rotation](../results/figures/06_rotation.png)

At 0°, 90° and 180° the CNN's accuracy and ECE are essentially flat — because the 90° rotations during training make these *exact* symmetries of the learned function. At 15° and 45° the accuracy drops by <3 pp because those angles are *not* in the augmentation distribution; interpolation artefacts and the fact that no training image was rotated by 15° break the symmetry slightly. **This is the cleanest possible case for ML robustness: train on a symmetry, get robustness to it for free.**

### 10.2 Blur: graceful degradation

![Blur](../results/figures/06_blur.png)

Accuracy drops from 0.55 to 0.40 as $\sigma$ goes from 0 to 8 px, but raw ECE actually **decreases** (0.11 → 0.05) — because the network correctly becomes *less confident* as the image loses detail. **Honest degradation.** The model knows it doesn't know.

### 10.3 Noise: sharp threshold

![Noise](../results/figures/06_noise.png)

Accuracy is roughly flat until $\sigma=25$, then collapses sharply to $0.41$ at $\sigma=50$. ECE only rises once the accuracy has already cratered — the model is *somewhat* over-confident in the new wrong regime, but not catastrophically.

### 10.4 Occlusion: confidently wrong

![Occlusion](../results/figures/06_occlusion.png)

**The pathological case.** At $s=32$ centred occlusion (a black square covering the central source), accuracy drops to $0.15$ — *barely above random for 10 classes* — while ECE triples to $0.33$. Mean confidence on wrong predictions actually *rises* from 0.36 (clean) to 0.47 (occluded). The network is confidently wrong, in exactly the way that should make a downstream science pipeline distrust its output.

## 11. Uncertainty under shift

Decomposing the ensemble's total entropy into **aleatoric** (each member's own entropy, averaged) and **epistemic** (Jensen gap between the entropy of the mean and the mean of the entropies) per Section 2.5:

In [7]:
shift = load('06b_uncertainty_under_shift.json')
print(f'T* fit on clean data: {shift["T_star"]:.3f}\n')
for pert, rows in shift['perturbations'].items():
    print(f'== {pert} ==')
    print(f'{"sev":>6}{"acc":>8}{"ECE_raw":>10}{"ECE_T*":>10}{"epist":>10}{"aleat":>10}')
    for r in rows:
        print(f'{r["severity"]:>6}{r["acc"]:>8.3f}{r["ece_raw"]:>10.3f}'
              f'{r["ece_T_scaled"]:>10.3f}{r["entropy_epistemic"]:>10.3f}'
              f'{r["entropy_aleatoric"]:>10.3f}')
    print()

T* fit on clean data: 0.728

== blur ==
   sev     acc   ECE_raw    ECE_T*     epist     aleat
   0.0   0.553     0.112     0.039     0.051     1.444
   1.0   0.551     0.115     0.043     0.047     1.459
   2.0   0.550     0.118     0.039     0.048     1.470
   4.0   0.517     0.105     0.032     0.053     1.516
   8.0   0.396     0.051     0.048     0.075     1.627

== noise ==
   sev     acc   ECE_raw    ECE_T*     epist     aleat
   0.0   0.553     0.112     0.039     0.051     1.444
  10.0   0.550     0.111     0.037     0.049     1.453
  25.0   0.528     0.090     0.023     0.056     1.447
  50.0   0.409     0.053     0.116     0.105     1.396
 100.0   0.226     0.123     0.190     0.196     1.674

== occlusion ==
   sev     acc   ECE_raw    ECE_T*     epist     aleat
   0.0   0.553     0.112     0.039     0.051     1.444
  32.0   0.145     0.333     0.428     0.082     1.372
  64.0   0.096     0.374     0.457     0.157     1.193
  96.0   0.076     0.327     0.394     0.284     1

![Uncertainty under shift](../results/figures/06b_uncertainty_under_shift.png)

**Three observations, each load-bearing for the conclusion:**

1. **Epistemic entropy correctly spikes on the hard shifts.** Under blur and rotation epistemic entropy stays around 0.05 — the ensemble members agree, because these corruptions look like the training distribution. Under occlusion it rises ~6× (0.05 → 0.32) and under heavy noise ~4× (0.05 → 0.20). *Even when the mean-softmax is miscalibrated, the disagreement between members correctly flags the OOD shifts.* This is the deep-ensemble payoff that motivated Lakshminarayanan et al. (2017).
2. **Aleatoric entropy is roughly flat across all conditions.** As it should be: each individual network is roughly equally confident under each condition, even when it's wrong. The aleatoric/epistemic split is doing its job of separating *irreducible* uncertainty from *model* uncertainty.
3. **Clean-data temperature scaling fails under shift.** $T^\star = 0.73$ fit on clean data cuts ECE from 0.11 to 0.04 on clean data (great!) and stays around 0.03–0.04 under blur and mild noise. But at heavy occlusion T-scaled ECE *rises* to 0.43 (vs 0.33 raw) — sharpening a model that is already wrong moves it further from the diagonal. This is precisely the Ovadia et al. (2019) finding: **scalar post-hoc fixes are not robust to distribution shift.** Per-input mechanisms (ensemble disagreement) are.

## 12. Limitations

Being honest about what this project does *not* establish:

1. **The perturbation sweep uses a reduced CNN** (32×32 inputs, $M=2$ members, 5 epochs) rather than the headline ensemble ($64\times64$, $M=5$, early-stopped) — a single $4\times5$ grid would otherwise cost ~20 hours of compute. The *trends* (epistemic-vs-aleatoric decomposition, occlusion-as-pathological, temperature-scaling-fails-under-shift) are structural properties we expect to be robust to scale, but the *absolute* ECE numbers in Section 10 will differ from those of the full ensemble.
2. **Pixel-level corruptions are a proxy for instrument-level shift.** A model trained on DECaLS that is deployed on, say, Euclid data faces a more complex shift (different filters, different PSF, different sky background statistics) than any of our four families simulates. Pixel corruptions are a tractable *lower bound* on the calibration problem; the true picture under instrument shift is likely worse.
3. **Inner $k=3$ shows visible variance in the inner macro-F1** (~0.01). A larger inner $k$ would tighten HP selection at non-trivial compute cost; we accepted the variance to keep the nested CV tractable on a laptop.
4. **Labels are themselves noisy** (Galaxy Zoo majority votes), particularly for the spiral subclasses and the boundary between Disturbed / Merging. Some of what we are calling "miscalibration" is the model confidently predicting one plausible label when another plausible label was the recorded one. The failure gallery (Section 9.5) makes this concrete.
5. **The deep ensemble has only $M=5$ members.** Members beyond ~5 keep improving calibration with diminishing returns (Lakshminarayanan et al. 2017); we cap at 5 for compute budget, not because the curve has plateaued.
6. **Class-weighted training interacts with calibration** in a way that pulls the ensemble into systematic under-confidence on majority classes — the Round Smooth ECE of 0.30 is largely this effect, not a generic model failure. Whether to weight is itself a calibration-vs-fairness trade-off this project does not adjudicate.
7. **We did not run MC-Dropout as a head-to-head comparison.** It is theoretically equivalent to ensembles in principle but typically weaker in practice on imbalanced multi-class problems; we argued for skipping it in the report's *Alternatives Considered* but did not empirically verify on this dataset.
8. **The metrics implementations are bin-based.** ECE with $B=15$ has known issues (sharp bin edges, sensitivity to $B$); a kernel-density variant or the threshold-free Brier decomposition would be more robust. We chose ECE for comparability with the calibration literature (Guo et al. 2017).

## 13. Conclusion

We asked whether clean-data calibration rankings are preserved under shift. **They are not.** The story in one paragraph:

> On clean data, the deep ensemble CNN wins on accuracy, macro-F1 and NLL, but the regularised logistic regression has the lowest ECE — the structurally simpler model is also the structurally better-calibrated one. Under shift, rotation and blur degrade the CNN honestly (it becomes correctly less confident), while occlusion and heavy noise produce confidently-wrong predictions whose mean-softmax ECE more than triples. **But the ensemble's member-disagreement — the epistemic component of its predictive entropy — rises 6× precisely on those shifts.** The right uncertainty signal is in the spread, not in the mean. A single scalar temperature fit on clean data sharpens calibration on near-distribution corruptions and *worsens* it on far-distribution ones, by construction.

For a downstream pipeline consuming galaxy-classification probabilities, this argues for:

1. **Deep ensembles over single networks**, not for the accuracy gain (modest) but for the access to a per-input epistemic signal that no single network can provide.
2. **Per-input thresholds on epistemic entropy**, not on max-softmax, when deciding which predictions to forward to downstream analysis.
3. **Explicit OOD detectors tuned to occlusion-like corruptions**, because post-hoc calibration alone provably cannot handle them.

These conclusions align with the broader cross-domain findings of Ovadia et al. (2019), reproduced here on a 10-class astronomical morphology task where the corruption families have direct physical interpretations.

---

### Reproducibility

Every figure in this notebook is regenerated by running `scripts/<NN>_*.py` from the project root. Full pipeline:

```bash
uv sync
# place Galaxy10_DECals.h5 in data/raw/ (see README)
for i in 01_eda 02_features 03_logreg 04_rf 05_cnn \
         05b_analysis 05c_temperature_scaling 05d_uncertainty_decomposition \
         05e_failure_analysis 05f_boundary_cases 05g_per_class_ece \
         06_perturbation_robustness 06b_uncertainty_under_shift ; do
  uv run python scripts/${i}.py
done
```

Total wall-clock on an Apple M-series laptop: ~4 hours, dominated by the full CNN nested CV.

### References

- **Lakshminarayanan, Pritzel & Blundell (2017).** *Simple and scalable predictive uncertainty estimation using deep ensembles.* NeurIPS.
- **Guo, Pleiss, Sun & Weinberger (2017).** *On calibration of modern neural networks.* ICML.
- **Ovadia et al. (2019).** *Can you trust your model's uncertainty? Evaluating predictive uncertainty under dataset shift.* NeurIPS.
- **Hendrycks & Dietterich (2019).** *Benchmarking neural network robustness to common corruptions and perturbations.* ICLR.
- **Leung & Bovy (2019).** Galaxy10 DECaLS dataset. *MNRAS* 483(3).
- **Conselice (2003)**, **Lotz, Primack & Madau (2004)** — CAS and Gini–M₂₀ morphology statistics.